# Análise dos microbenchmarks dos componentes Edge AI

Este notebook analisa os relatórios produzidos por `benchmarks/benchmark_components.py` para uso em relatório técnico ou artigo científico.

Os microbenchmarks são **1.000 execuções sequenciais e isoladas** de cada etapa. Portanto, medem tempo de serviço sob condições controladas: não incluem concorrência entre módulos, espera em filas ou sobreposição de processamento. Os resultados não devem ser interpretados como throughput fim a fim do pipeline.

A análise cobre: distribuição e estatísticas de latência; estabilidade, outliers e drift; aquecimento e ambiente; capacidade isolada; contribuição relativa das etapas; projeções com FPS e proporção de frames adequados; e implicações para a interpretação do pipeline completo.

## Método e convenções

- `total_stage_ms` é a métrica principal de cada componente.
- Para selector e predictor, `tflite_total_ms` contém `set_tensor + invoke + get_tensor`; `invoke_ms` é a inferência isolada.
- Enhancement é medido pela função real `DataEnhance.run`; predictor recebe o tensor já enhanced.
- A capacidade isolada é `1000 / tempo_médio_ms` e representa operações por segundo em serviço isolado.
- A projeção de FPS usa o modelo de demanda `FPS × [S + q(E + P)]`, onde `q` é a proporção de frames adequados. É uma projeção de demanda computacional, não uma reprodução da concorrência do pipeline.
- Outliers são reportados, nunca removidos dos resultados principais. A análise sem outliers IQR é secundária.

In [ ]:
from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib
try:
    from IPython import get_ipython
    if get_ipython() is not None:
        get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
    'pdf.fonttype': 42,
})

# Relatório fixado explicitamente para a revisão visual compacta.
REPORT_DIR = Path('benchmarks/benchmark_runs/all_20260713_180430/benchmark_components_2026-07-13_18-04-49')
REPORT_ROOT_CANDIDATES = [
    Path('../benchmarks/runs'),
    Path('../benchmarks/benchmark_runs'),
    Path('benchmarks/runs'),
    Path('benchmarks/benchmark_runs'),
    Path('../benchmark_runs'),
    Path('benchmark_runs'),
]
OUTPUT_DIR_NAME = 'analysis_components'
N_EXPECTED = 1000
BOOTSTRAP_RESAMPLES = 10000
SEED = 42

def find_report_dir():
    if REPORT_DIR is not None:
        p = Path(REPORT_DIR).expanduser()
        if not p.is_absolute():
            for base in [Path.cwd(), Path.cwd() / '..']:
                candidate = (base / p).resolve()
                if candidate.exists():
                    return candidate
        return p.resolve()
    candidates = []
    for root in REPORT_ROOT_CANDIDATES:
        root = root.resolve()
        if root.exists():
            candidates.extend(p for p in root.glob('**/benchmark_components_*')
                              if (p / 'metadata.json').exists()
                              and (p / 'summary.json').exists())
    if not candidates:
        raise FileNotFoundError('Nenhum relatório benchmark_components_* encontrado.')
    return max(candidates, key=lambda p: p.stat().st_mtime)

REPORT_DIR = find_report_dir()
ANALYSIS_DIR = REPORT_DIR / OUTPUT_DIR_NAME
ANALYSIS_DIR.mkdir(exist_ok=True)
print('Relatório:', REPORT_DIR)
print('Saída da análise:', ANALYSIS_DIR)

## 1. Carregamento, validação e inventário do relatório

In [ ]:
with open(REPORT_DIR / 'metadata.json', encoding='utf-8') as f:
    metadata = json.load(f)
with open(REPORT_DIR / 'summary.json', encoding='utf-8') as f:
    summary_json = json.load(f)

def read_measurements(component):
    path = REPORT_DIR / f'{component}_measurements.csv'
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    df['component'] = component
    return df

components = ['selector', 'enhancer', 'predictor', 'aggregation']
measurements = {c: read_measurements(c) for c in components}
warmups = {}
for c in components:
    path = REPORT_DIR / f'warmup_{c}.csv'
    warmups[c] = pd.read_csv(path) if path.exists() else pd.DataFrame()

monitor_path = REPORT_DIR / 'system_monitor.csv'
monitor = pd.read_csv(monitor_path) if monitor_path.exists() else pd.DataFrame()
failures_path = REPORT_DIR / 'failures.csv'
failures = pd.read_csv(failures_path) if failures_path.exists() else pd.DataFrame()

for c, df in measurements.items():
    for col in df.columns:
        if col.endswith('_ns') or col.endswith('_ms') or col in [
            'iteration', 'score', 'prediction', 'result', 'num_predictions',
            'predicted_argmax', 'output_min', 'output_max', 'output_mean',
            'output_std', 'timestamp_monotonic_ns']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

config = metadata.get('config', {})
print('Componente | medições | warm-ups | falhas')
print('-' * 48)
for c in components:
    n_failures = int((failures['component'] == c).sum()) if 'component' in failures else 0
    print(f'{c:11s} | {len(measurements[c]):9d} | {len(warmups[c]):8d} | {n_failures:6d}')
print('\nConfiguração:', config)
print('Temperatura inicial:', metadata.get('environment_before', {}).get('temperature_celsius'))
print('Temperatura final:', metadata.get('environment_after', {}).get('temperature_celsius'))

In [ ]:
# Validações metodológicas importantes. A agregação possui 5 tamanhos,
# portanto normalmente tem 5 x N_EXPECTED observações.
validation = []
for c in ['selector', 'enhancer', 'predictor']:
    n = len(measurements[c])
    validation.append({'component': c, 'expected': N_EXPECTED,
                      'observed': n, 'ok': n == N_EXPECTED})
validation.append({'component': 'aggregation', 'expected': '5 x iterations',
                  'observed': len(measurements['aggregation']),
                  'ok': len(measurements['aggregation']) >= N_EXPECTED})
validation_df = pd.DataFrame(validation)
display(validation_df)

if not validation_df.ok.all():
    print('ATENÇÃO: há componentes com número de observações diferente do protocolo.')
if not failures.empty:
    display(failures.head())
else:
    print('Nenhuma falha registrada.')

## 2. Estatísticas descritivas e capacidade isolada

A tabela principal usa o tempo total de serviço de cada módulo. Para selector e predictor, as decomposições TFLite aparecem em uma tabela separada.

In [ ]:
MAIN_METRICS = {
    'selector': 'total_stage_ms',
    'enhancer': 'total_stage_ms',
    'predictor': 'total_stage_ms',
    'aggregation': 'aggregation_ms',
}
LABELS = {
    'selector': 'Seletor de frames',
    'enhancer': 'Enhancement',
    'predictor': 'Preditor de peso',
    'aggregation': 'Agregação',
}

def finite_values(df, metric):
    return pd.to_numeric(df.get(metric, pd.Series(dtype=float)),
                         errors='coerce').dropna().to_numpy(dtype=float)

def bootstrap_stat_ci(values, statistic, n_boot=BOOTSTRAP_RESAMPLES,
                      seed=SEED, confidence=0.95, chunk=250):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 2:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    boot_stats = []
    for start in range(0, n_boot, chunk):
        size = min(chunk, n_boot - start)
        indices = rng.integers(0, len(values), size=(size, len(values)))
        sample = values[indices]
        if statistic == 'median':
            boot_stats.append(np.median(sample, axis=1))
        else:
            boot_stats.append(np.percentile(sample, float(statistic), axis=1))
    boot_stats = np.concatenate(boot_stats)
    alpha = (1 - confidence) / 2
    return tuple(np.quantile(boot_stats, [alpha, 1 - alpha]))

def describe_latency(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n == 0:
        return {}
    mean = float(np.mean(values))
    median = float(np.median(values))
    sd = float(np.std(values, ddof=1)) if n > 1 else 0.0
    q1, q3 = np.percentile(values, [25, 75])
    iqr = q3 - q1
    fence_low, fence_high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    if n > 1:
        ci_mean = stats.t.interval(0.95, df=n - 1, loc=mean,
                                  scale=stats.sem(values))
    else:
        ci_mean = (np.nan, np.nan)
    med_ci = bootstrap_stat_ci(values, 'median')
    p95_ci = bootstrap_stat_ci(values, 95)
    p99_ci = bootstrap_stat_ci(values, 99)
    return {
        'n': n, 'mean_ms': mean, 'median_ms': median,
        'std_ms': sd, 'variance_ms2': float(np.var(values, ddof=1)) if n > 1 else 0.0,
        'cv': sd / mean if mean else np.nan,
        'sem_ms': sd / np.sqrt(n) if n else np.nan,
        'ci_mean_low_ms': ci_mean[0], 'ci_mean_high_ms': ci_mean[1],
        'bootstrap_median_low_ms': med_ci[0], 'bootstrap_median_high_ms': med_ci[1],
        'p90_ms': float(np.percentile(values, 90)),
        'p95_ms': float(np.percentile(values, 95)),
        'p99_ms': float(np.percentile(values, 99)),
        'bootstrap_p95_low_ms': p95_ci[0], 'bootstrap_p95_high_ms': p95_ci[1],
        'bootstrap_p99_low_ms': p99_ci[0], 'bootstrap_p99_high_ms': p99_ci[1],
        'min_ms': float(np.min(values)), 'max_ms': float(np.max(values)),
        'range_ms': float(np.ptp(values)), 'iqr_ms': float(iqr),
        'mad_ms': float(np.median(np.abs(values - median))),
        'skewness': float(stats.skew(values, bias=False)) if n > 2 else np.nan,
        'kurtosis_excess': float(stats.kurtosis(values, bias=False)) if n > 3 else np.nan,
        'n_outliers_iqr': int(np.sum((values < fence_low) | (values > fence_high))),
        'throughput_by_mean_ops_s': 1000 / mean if mean > 0 else np.nan,
        'throughput_by_median_ops_s': 1000 / median if median > 0 else np.nan,
    }

main_rows = []
for component, metric in MAIN_METRICS.items():
    row = describe_latency(finite_values(measurements[component], metric))
    row.update(component=component, stage=LABELS[component], metric=metric)
    main_rows.append(row)
main_stats = pd.DataFrame(main_rows).set_index('component')
display(main_stats.round(4))

In [ ]:
# Decomposição do custo dos modelos: total, TFLite total e invoke.
decomp_rows = []
for component in ['selector', 'predictor']:
    for metric in ['total_stage_ms', 'tflite_total_ms', 'invoke_ms']:
        row = describe_latency(finite_values(measurements[component], metric))
        row.update(component=component, stage=LABELS[component], metric=metric)
        decomp_rows.append(row)
decomp_stats = pd.DataFrame(decomp_rows)
display(decomp_stats[['stage', 'metric', 'n', 'mean_ms', 'median_ms', 'p95_ms', 'p99_ms', 'cv', 'min_ms', 'max_ms']].round(4))

## 3. Qual etapa é mais lenta e qual é mais estável?

In [ ]:
SERVICE_COMPONENTS = ['selector', 'enhancer', 'predictor']
ranked = main_stats.loc[SERVICE_COMPONENTS].sort_values('median_ms', ascending=False)
slowest = ranked.index[0]
most_stable = main_stats.loc[SERVICE_COMPONENTS, 'cv'].idxmin()
most_variable = main_stats.loc[SERVICE_COMPONENTS, 'cv'].idxmax()
print('Maior mediana de tempo: {} ({:.4f} ms)'.format(LABELS[slowest], ranked.loc[slowest, 'median_ms']))
print('Maior estabilidade relativa: {} (CV={:.4f})'.format(LABELS[most_stable], main_stats.loc[most_stable, 'cv']))
print('Maior variabilidade relativa: {} (CV={:.4f})'.format(LABELS[most_variable], main_stats.loc[most_variable, 'cv']))
display(ranked[['stage', 'mean_ms', 'median_ms', 'std_ms', 'cv', 'p95_ms', 'p99_ms']].round(4))

## 4. Distribuições de latência

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for ax, (component, metric) in zip(axes.flat, MAIN_METRICS.items()):
    values = finite_values(measurements[component], metric)
    ax.hist(values, bins=40, color='#2563eb', alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(values), color='#dc2626', label='média')
    ax.axvline(np.median(values), color='#16a34a', label='mediana')
    ax.set_title(LABELS[component])
    ax.set_xlabel('Tempo (ms)')
    ax.set_ylabel('Frequência')
    ax.legend()
fig.suptitle('Distribuição das latências isoladas', fontsize=14)
plt.savefig(ANALYSIS_DIR / 'latency_histograms.png', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
box_data = [finite_values(measurements[c], m) for c, m in MAIN_METRICS.items()]
ax.boxplot(box_data, tick_labels=[LABELS[c] for c in MAIN_METRICS], showfliers=True)
ax.set_ylabel('Tempo de serviço (ms)')
ax.set_title('Boxplot comparativo com outliers')
ax.tick_params(axis='x', rotation=20)
plt.savefig(ANALYSIS_DIR / 'latency_boxplot.png', bbox_inches='tight')
plt.show()

## 5. Outliers, drift e aquecimento

A sequência temporal é analisada sem ordenar por imagem. Isso permite identificar mudança de comportamento ao longo das 1.000 operações.

In [ ]:
def temporal_analysis(values, block_size=100):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    x = np.arange(1, len(values) + 1)
    slope = stats.linregress(x, values).slope if len(values) > 2 else np.nan
    first_n = min(100, len(values))
    first = values[:first_n]
    last = values[-first_n:]
    blocks = []
    for start in range(0, len(values), block_size):
        chunk = values[start:start + block_size]
        blocks.append({'block': start // block_size + 1,
                       'iteration_start': start + 1,
                       'iteration_end': start + len(chunk),
                       'mean_ms': np.mean(chunk),
                       'median_ms': np.median(chunk),
                       'p95_ms': np.percentile(chunk, 95)})
    return {'slope_ms_per_iteration': slope,
            'first_100_mean_ms': np.mean(first),
            'last_100_mean_ms': np.mean(last),
            'last_vs_first_pct': 100 * (np.mean(last) - np.mean(first)) / np.mean(first),
            'blocks': pd.DataFrame(blocks)}

temporal = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
for ax, (component, metric) in zip(axes.flat, MAIN_METRICS.items()):
    values = finite_values(measurements[component], metric)
    result = temporal_analysis(values)
    temporal[component] = result
    ax.plot(np.arange(1, len(values) + 1), values, alpha=0.25, linewidth=0.5)
    rolling = pd.Series(values).rolling(51, center=True, min_periods=1).median()
    ax.plot(rolling, color='#dc2626', linewidth=1.5, label='mediana móvel (51)')
    ax.set_title(LABELS[component])
    ax.set_xlabel('Iteração válida')
    ax.set_ylabel('ms')
    ax.legend()
fig.suptitle('Latência ao longo das execuções', fontsize=14)
plt.savefig(ANALYSIS_DIR / 'latency_over_iterations.png', bbox_inches='tight')
plt.show()

drift_rows = []
for c, result in temporal.items():
    drift_rows.append({'component': c, 'stage': LABELS[c],
                       **{k: v for k, v in result.items() if k != 'blocks'}})
drift_df = pd.DataFrame(drift_rows).set_index('component')
display(drift_df.round(6))

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for ax, (component, result) in zip(axes.flat, temporal.items()):
    blocks = result['blocks']
    ax.plot(blocks['block'], blocks['mean_ms'], marker='o', label='média')
    ax.plot(blocks['block'], blocks['p95_ms'], marker='o', label='P95')
    ax.set_title(LABELS[component])
    ax.set_xlabel('Bloco de 100 medições')
    ax.set_ylabel('ms')
    ax.legend()
fig.suptitle('Métricas por blocos de 100 execuções', fontsize=14)
plt.savefig(ANALYSIS_DIR / 'latency_blocks_100.png', bbox_inches='tight')
plt.show()

In [ ]:
# Comparação específica dos warm-ups com as primeiras medições válidas.
warmup_rows = []
for component, metric in MAIN_METRICS.items():
    w = finite_values(warmups[component], metric)
    v = finite_values(measurements[component], metric)[:len(w)]
    warmup_rows.append({
        'component': component, 'stage': LABELS[component],
        'warmup_n': len(w), 'warmup_mean_ms': np.mean(w) if len(w) else np.nan,
        'first_valid_n': len(v), 'first_valid_mean_ms': np.mean(v) if len(v) else np.nan,
        'difference_pct': 100 * (np.mean(v) - np.mean(w)) / np.mean(w)
        if len(w) and np.mean(w) else np.nan,
    })
warmup_comparison = pd.DataFrame(warmup_rows).set_index('component')
display(warmup_comparison.round(4))

## 6. Ambiente, temperatura e possíveis efeitos térmicos

In [ ]:
if monitor.empty:
    print('system_monitor.csv não disponível.')
else:
    monitor['timestamp_utc'] = pd.to_datetime(monitor['timestamp_utc'], errors='coerce')
    for col in ['cpu_total_percent', 'process_cpu_percent', 'process_rss_bytes',
                'temperature_celsius', 'cpu_freq_cur_hz']:
        if col in monitor:
            monitor[col] = pd.to_numeric(monitor[col], errors='coerce')
    monitor['component_clean'] = monitor['component'].astype(str).str.extract(
        r'^(selector|enhancer|predictor|aggregation)', expand=False)
    monitor['component_clean'] = monitor['component_clean'].fillna(monitor['component'])
    env_rows = []
    for component, g in monitor.groupby('component_clean'):
        row = {'component': component, 'stage': LABELS.get(component, component)}
        for col in ['temperature_celsius', 'cpu_total_percent',
                    'process_cpu_percent', 'cpu_freq_cur_hz']:
            if col in g:
                row[col + '_mean'] = g[col].mean()
                row[col + '_max'] = g[col].max()
        env_rows.append(row)
    env_summary = pd.DataFrame(env_rows).set_index('component')
    display(env_summary.round(3))
    
    throttle_cols = [c for c in ['throttled_now', 'freq_capped_now', 'undervoltage_now']
                     if c in monitor.columns]
    if throttle_cols:
        display(monitor[throttle_cols].astype(str).apply(lambda s: s.value_counts()))

    fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, constrained_layout=True)
    if 'temperature_celsius' in monitor:
        axes[0].plot(monitor['timestamp_utc'], monitor['temperature_celsius'], color='#dc2626')
        axes[0].axhline(80, color='#f59e0b', linestyle='--', label='80 °C')
        axes[0].axhline(85, color='#991b1b', linestyle='--', label='85 °C')
        axes[0].set_ylabel('Temperatura (°C)')
        axes[0].legend()
    if 'process_cpu_percent' in monitor:
        axes[1].plot(monitor['timestamp_utc'], monitor['process_cpu_percent'], color='#2563eb')
        axes[1].set_ylabel('CPU do processo (%)')
    if 'cpu_freq_cur_hz' in monitor:
        axes[2].plot(monitor['timestamp_utc'], monitor['cpu_freq_cur_hz'] / 1e9, color='#16a34a')
        axes[2].set_ylabel('Frequência (GHz)')
    axes[2].set_xlabel('Tempo UTC')
    fig.suptitle('Monitoramento do ambiente durante o benchmark')
    plt.savefig(ANALYSIS_DIR / 'environment_monitor.png', bbox_inches='tight')
    plt.show()

## 7. Capacidade isolada e participação no custo de um frame adequado

In [ ]:
# Para a composição de um frame adequado, excluímos agregação por ser
# feita uma vez por conjunto/passageiro, não por frame.
frame_components = ['selector', 'enhancer', 'predictor']
composition = main_stats.loc[frame_components, ['mean_ms', 'median_ms', 'p95_ms', 'p99_ms']].copy()
composition['share_mean_pct'] = 100 * composition['mean_ms'] / composition['mean_ms'].sum()
composition['isolated_capacity_ops_s_mean'] = 1000 / composition['mean_ms']
composition['isolated_capacity_ops_s_median'] = 1000 / composition['median_ms']
total_frame_mean_ms = composition['mean_ms'].sum()
composition.loc['TOTAL', 'mean_ms'] = total_frame_mean_ms
composition.loc['TOTAL', 'share_mean_pct'] = 100.0
composition.loc['TOTAL', 'isolated_capacity_ops_s_mean'] = 1000 / total_frame_mean_ms
display(composition.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
shares = composition.loc[frame_components, 'share_mean_pct']
ax.bar([LABELS[c] for c in frame_components], shares, color=['#2563eb', '#16a34a', '#dc2626'])
ax.set_ylabel('Participação no tempo total (%)')
ax.set_title('Custo relativo por frame adequado — serviço isolado')
ax.tick_params(axis='x', rotation=20)
for i, value in enumerate(shares):
    ax.text(i, value, f'{value:.1f}%', ha='center', va='bottom')
plt.savefig(ANALYSIS_DIR / 'component_shares.png', bbox_inches='tight')
plt.show()

## 8. Projeção para FPS e proporção de frames adequados

Se o seletor processa todos os frames e apenas a proporção `q` chega ao enhancement e ao predictor, a demanda de serviço isolado por segundo é:

$$D(FPS,q) = FPS × [S + q(E + P)] / 1000$$

onde `D` é o número equivalente de segundos de serviço por segundo de entrada. `D = 1` corresponde a 100% de um recurso serial hipotético. No pipeline real, os módulos são concorrentes e cada estágio tem sua própria fila/worker; portanto esta figura é uma referência de carga, não uma previsão exata de throughput.

In [ ]:
S = main_stats.loc['selector', 'mean_ms']
E = main_stats.loc['enhancer', 'mean_ms']
P = main_stats.loc['predictor', 'mean_ms']
fps_values = np.array([1, 2, 5, 10, 15, 20, 30, 60])
q_values = np.array([0.10, 0.25, 0.50, 0.75, 0.90, 1.00])
projection_rows = []
for fps in fps_values:
    for q in q_values:
        service_ms_per_input_frame = S + q * (E + P)
        demand = fps * service_ms_per_input_frame / 1000
        projection_rows.append({
            'fps': fps, 'suitable_fraction': q,
            'service_ms_per_input_frame': service_ms_per_input_frame,
            'service_demand_s_per_s': demand,
            'demand_percent': 100 * demand,
            'ideal_serial_input_capacity_fps': 1000 / service_ms_per_input_frame,
            'selector_demand_percent': 100 * fps * S / 1000,
            'enhancer_demand_percent': 100 * fps * q * E / 1000,
            'predictor_demand_percent': 100 * fps * q * P / 1000,
        })
projection = pd.DataFrame(projection_rows)
display(projection.head(12).round(4))
projection.to_csv(ANALYSIS_DIR / 'fps_suitable_fraction_projection.csv', index=False)

pivot = projection.pivot(index='suitable_fraction', columns='fps', values='demand_percent')
fig, ax = plt.subplots(figsize=(11, 4.8))
im = ax.imshow(pivot.values, aspect='auto', cmap='magma', origin='lower')
ax.set_xticks(range(len(pivot.columns)), labels=pivot.columns)
ax.set_yticks(range(len(pivot.index)), labels=[f'{100*q:.0f}%' for q in pivot.index])
ax.set_xlabel('FPS de entrada')
ax.set_ylabel('Proporção suited')
ax.set_title('Demanda de serviço isolado projetada (%)')
fig.colorbar(im, ax=ax, label='segundos de serviço por segundo × 100')
plt.savefig(ANALYSIS_DIR / 'fps_suitable_fraction_heatmap.png', bbox_inches='tight')
plt.show()

# Limite isolado aproximado quando D=1, para cada q.
capacity_by_q = pd.DataFrame({
    'suitable_fraction': q_values,
    'ideal_serial_capacity_fps': 1000 / (S + q_values * (E + P)),
})
display(capacity_by_q.round(4))

## 9. Interpretação científica dos resultados

A célula seguinte gera um texto-base para o artigo. Ele deve ser revisado junto com os gráficos, principalmente quando houver sinais de throttling, falhas ou número de medições diferente de 1.000.

In [ ]:
def fmt(value, digits=3):
    return 'NA' if pd.isna(value) else f'{value:.{digits}f}'

print('INTERPRETAÇÃO PARA O ARTIGO')
print('')
print('O maior tempo de serviço isolado foi observado em {}, com média de {:.3f} ms e mediana de {:.3f} ms.'.format(LABELS[slowest], main_stats.loc[slowest, 'mean_ms'], main_stats.loc[slowest, 'median_ms']))
print('A capacidade mediana estimada desse estágio é {:.3f} operações/s.'.format(main_stats.loc[slowest, 'throughput_by_median_ops_s']))
print('O componente mais estável foi {} (CV={:.4f}); o mais variável foi {} (CV={:.4f}).'.format(LABELS[most_stable], main_stats.loc[most_stable, 'cv'], LABELS[most_variable], main_stats.loc[most_variable, 'cv']))
print('')
for component, result in temporal.items():
    pct = result['last_vs_first_pct']
    direction = 'aumento' if pct >= 0 else 'redução'
    print('{}: {} de {:.2f}% entre as médias das primeiras e últimas 100 medições; inclinação={:.6g} ms/iteração.'.format(LABELS[component], direction, abs(pct), result['slope_ms_per_iteration']))
print('')
print('A projeção FPS × suited mostra que o seletor gera demanda proporcional a todos os frames, '
      'enquanto enhancement e predictor escalam com a fração suited q.')
print('Esses resultados explicam a pressão de serviço esperada, mas não substituem a latência fim a fim: '
      'o pipeline real possui threads, filas, buffers e sobreposição entre estágios.')

interpretation_text = (
    'Os tempos apresentados caracterizam o tempo de serviço isolado dos componentes do pipeline Edge AI. '
    'A extrapolação para FPS deve ser interpretada como demanda computacional esperada: o seletor opera '
    'sobre todos os frames, enquanto enhancement e predição operam apenas sobre frames classificados como '
    'adequados. Assim, a proporção suited controla diretamente a parcela de custo downstream. '
    'A comparação com a execução completa deve considerar concorrência, filas, aquecimento, throttling, '
    'tempo de captura e agregação, que não estão contidos neste microbenchmark.'
)
(ANALYSIS_DIR / 'interpretation_base.txt').write_text(interpretation_text, encoding='utf-8')

## 10. Exportação para artigo

São exportadas tabelas CSV, Markdown e LaTeX com as estatísticas principais e a decomposição dos modelos. Os gráficos PNG ficam no diretório `analysis_components` dentro do relatório.

In [ ]:
# Compact one-column IEEE article table.
import shutil
import subprocess
import tempfile

ARTICLE_COMPONENTS = ['selector', 'enhancer', 'predictor']
ARTICLE_STAGE_NAMES = {
    'selector': 'Selection',
    'enhancer': 'Enhancement',
    'predictor': 'Prediction',
}
TARGET_WIDTH_IN = 3.50
TABLE_FONT_SIZE = r'\footnotesize'
TABLE_TABCOLSEP_PT = 2.3
TABLE_ARRAYSTRETCH = 1.03
PREVIOUS_ARRAYSTRETCH = 1.08

# Resolve the repository without relying on the notebook launch directory.
repo_candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT_COMPACT = next(
    candidate for candidate in repo_candidates
    if (candidate / 'article_artifacts_compact/validation/microbenchmark_numeric_baseline.csv').exists()
)
COMPACT_TABLE_DIR = REPO_ROOT_COMPACT / 'article_artifacts_compact/tables'
COMPACT_RENDER_DIR = REPO_ROOT_COMPACT / 'article_artifacts_compact/render_comparison'
MICRO_BASELINE_PATH = (
    REPO_ROOT_COMPACT
    / 'article_artifacts_compact/validation/microbenchmark_numeric_baseline.csv'
)
COMPACT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
COMPACT_RENDER_DIR.mkdir(parents=True, exist_ok=True)

# All values originate from main_stats, calculated by the unchanged statistics cell.
article_table = (
    main_stats.loc[
        ARTICLE_COMPONENTS,
        ['mean_ms', 'ci_mean_low_ms', 'ci_mean_high_ms'],
    ]
    .reset_index()
    .rename(columns={
        'index': 'component',
        'ci_mean_low_ms': 'mean_ci95_low_ms',
        'ci_mean_high_ms': 'mean_ci95_high_ms',
    })
)
article_table['stage'] = article_table['component'].map(ARTICLE_STAGE_NAMES)
article_table['theoretical_capacity_ops_s'] = 1000.0 / article_table['mean_ms']
article_table = article_table[
    [
        'component',
        'stage',
        'mean_ms',
        'mean_ci95_low_ms',
        'mean_ci95_high_ms',
        'theoretical_capacity_ops_s',
    ]
]

# Assertions against the immutable numeric baseline.
microbenchmark_baseline = pd.read_csv(
    MICRO_BASELINE_PATH, float_precision='round_trip'
).set_index('component')
baseline_rows = microbenchmark_baseline.loc[ARTICLE_COMPONENTS]
assert article_table['component'].tolist() == ARTICLE_COMPONENTS
assert article_table['stage'].tolist() == ['Selection', 'Enhancement', 'Prediction']
assert 'aggregation' not in set(article_table['component'])
assert len(article_table) == 3
np.testing.assert_array_equal(
    article_table['mean_ms'].to_numpy(dtype=float),
    baseline_rows['mean_ms'].to_numpy(dtype=float),
)
np.testing.assert_array_equal(
    article_table['mean_ci95_low_ms'].to_numpy(dtype=float),
    baseline_rows['mean_ci95_low_ms'].to_numpy(dtype=float),
)
np.testing.assert_array_equal(
    article_table['mean_ci95_high_ms'].to_numpy(dtype=float),
    baseline_rows['mean_ci95_high_ms'].to_numpy(dtype=float),
)
np.testing.assert_array_equal(
    article_table['theoretical_capacity_ops_s'].to_numpy(dtype=float),
    baseline_rows['theoretical_capacity_by_mean_ops_s'].to_numpy(dtype=float),
)
np.testing.assert_array_equal(
    article_table['theoretical_capacity_ops_s'].to_numpy(dtype=float),
    1000.0 / article_table['mean_ms'].to_numpy(dtype=float),
)
assert np.isfinite(
    article_table[
        [
            'mean_ms',
            'mean_ci95_low_ms',
            'mean_ci95_high_ms',
            'theoretical_capacity_ops_s',
        ]
    ].to_numpy(dtype=float)
).all()

# Expected display values are validation targets only; table cells use computed values.
expected_display = {
    'Selection': ('14.552', '[14.541--14.562]', '68.72'),
    'Enhancement': ('5.909', '[5.902--5.916]', '169.23'),
    'Prediction': ('136.413', '[136.396--136.431]', '7.33'),
}
computed_display = {
    row.stage: (
        f'{row.mean_ms:.3f}',
        f'[{row.mean_ci95_low_ms:.3f}--{row.mean_ci95_high_ms:.3f}]',
        f'{row.theoretical_capacity_ops_s:.2f}',
    )
    for row in article_table.itertuples(index=False)
}
assert computed_display == expected_display

latex_rows = [
    (
        f'{row.stage} & {row.mean_ms:.3f} & '
        f'[{row.mean_ci95_low_ms:.3f}--{row.mean_ci95_high_ms:.3f}] & '
        f'{row.theoretical_capacity_ops_s:.2f} \\\\'
    )
    for row in article_table.itertuples(index=False)
]
latex_text = '\n'.join([
    r'\begin{table}[t]',
    r'\centering',
    r'\caption{Isolated component microbenchmark results.}',
    r'\label{tab:component_microbenchmarks}',
    TABLE_FONT_SIZE,
    rf'\setlength{{\tabcolsep}}{{{TABLE_TABCOLSEP_PT:.1f}pt}}',
    rf'\renewcommand{{\arraystretch}}{{{TABLE_ARRAYSTRETCH:.2f}}}',
    r'\begin{tabular*}{\columnwidth}{@{\extracolsep{\fill}}lccc@{}}',
    r'\toprule',
    (
        r'\textbf{Component} & '
        r'\shortstack{\textbf{Mean}\\\textbf{(ms)}} & '
        r'\shortstack{\textbf{95\% CI}\\\textbf{(ms)}} & '
        r'\shortstack{\textbf{Theor. capacity}\\\textbf{(ops/s)}} \\'
    ),
    r'\midrule',
    *latex_rows,
    r'\bottomrule',
    r'\end{tabular*}',
    r'\end{table}',
    '',
])
assert r'\resizebox' not in latex_text
assert r'\begin{adjustbox}' not in latex_text
assert r'\scriptsize' not in latex_text
assert r'\footnotesize' in latex_text
assert 'theoretical_capacity_by_median' not in latex_text

compact_tex_path = COMPACT_TABLE_DIR / 'tab_component_microbenchmarks_compact.tex'
if compact_tex_path.exists():
    raise FileExistsError(f'Refusing to overwrite existing compact table: {compact_tex_path}')
compact_tex_path.write_text(latex_text, encoding='utf-8')

# Conditional IEEEtran compilation. No package is installed or modified here.
pdflatex_path = shutil.which('pdflatex')
compact_pdf_path = (
    COMPACT_RENDER_DIR / 'tab_component_microbenchmarks_compact_test.pdf'
)
latex_test_result = {
    'compiler': pdflatex_path,
    'compiled': False,
    'pdf_path': None,
    'overfull_boxes': [],
}
if pdflatex_path:
    with tempfile.TemporaryDirectory(prefix='microbenchmark_table_ieee_') as temp_dir:
        temp_dir = Path(temp_dir)
        test_tex = temp_dir / 'table_test.tex'
        test_tex.write_text(
            '\n'.join([
                r'\documentclass[conference]{IEEEtran}',
                r'\usepackage{booktabs}',
                r'\begin{document}',
                rf'\input{{{compact_tex_path.as_posix()}}}',
                r'\end{document}',
                '',
            ]),
            encoding='utf-8',
        )
        completed = subprocess.run(
            [
                pdflatex_path,
                '-interaction=nonstopmode',
                '-halt-on-error',
                test_tex.name,
            ],
            cwd=temp_dir,
            capture_output=True,
            text=True,
            check=False,
        )
        log_path = temp_dir / 'table_test.log'
        log_text = (
            log_path.read_text(encoding='utf-8', errors='replace')
            if log_path.exists()
            else completed.stdout + completed.stderr
        )
        overfull_boxes = [
            line.strip() for line in log_text.splitlines()
            if 'Overfull' in line
        ]
        if completed.returncode != 0:
            raise RuntimeError(
                f'IEEEtran test compilation failed ({completed.returncode}).\n{log_text}'
            )
        generated_pdf = temp_dir / 'table_test.pdf'
        if not generated_pdf.exists():
            raise FileNotFoundError(generated_pdf)
        if compact_pdf_path.exists():
            raise FileExistsError(
                f'Refusing to overwrite existing compact test PDF: {compact_pdf_path}'
            )
        shutil.copy2(generated_pdf, compact_pdf_path)
        latex_test_result = {
            'compiler': pdflatex_path,
            'compiled': True,
            'pdf_path': str(compact_pdf_path),
            'overfull_boxes': overfull_boxes,
        }

assertion_results = {
    'mean_equals_baseline': True,
    'mean_ci95_equals_baseline': True,
    'capacity_equals_1000_over_mean': True,
    'three_required_rows_present': True,
    'aggregation_absent': True,
    'no_nonfinite_values': True,
}
estimated_height_reduction_percent = (
    100.0 * (1.0 - TABLE_ARRAYSTRETCH / PREVIOUS_ARRAYSTRETCH)
)

print('Compact table:', compact_tex_path)
print('Target width: 3.50 in (IEEE column width)')
print('Font:', TABLE_FONT_SIZE)
print('tabcolsep:', f'{TABLE_TABCOLSEP_PT:.1f} pt')
print('arraystretch:', f'{TABLE_ARRAYSTRETCH:.2f}')
print(
    'Estimated height reduction from arraystretch only:',
    f'{estimated_height_reduction_percent:.2f}%',
)
print('Assertions:', assertion_results)
print('LaTeX test:', latex_test_result)
display(article_table)


## Limitações e cuidados para publicação

1. O benchmark mede serviço isolado, não a latência entre o último frame capturado e o peso final.
2. A projeção de FPS assume que a proporção suited `q` é conhecida e estacionária; na prática ela pode variar por animal, fazenda, posição e qualidade da captura.
3. A capacidade `ops/s` não deve ser apresentada como throughput fim a fim do sistema.
4. A análise térmica deve usar a temperatura, frequência e flags de throttling do relatório correspondente.
5. Se houver menos de 1.000 medições válidas, a conclusão deve registrar explicitamente a limitação.
6. Para o artigo, cite separadamente a metodologia do microbenchmark e os experimentos completos de FPS/native timestamps.